In [ ]:
# Install / upgrade all dependencies
#   NOTE: we uninstall first so there are no stale wheels conflicting.
#         Restart the runtime after this cell completes.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torch", "torchvision", "torchaudio",
                "transformers", "peft", "trl", "accelerate", "bitsandbytes"],
               capture_output=True)

# Install PyTorch with CUDA 12.4 (matches Colab's driver in 2025)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu124"],
               check=True)

# Pin library versions that are known-good together with Phi-4-mini-instruct
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.48.1",
                "accelerate==1.3.0",
                "peft==0.14.0",
                "trl==0.15.2",      # processing_class API is stable here
                "bitsandbytes>=0.43.0",
                "datasets>=2.18.0",
                "scikit-learn",
                "pandas",
                "numpy",
                "sentencepiece",
                "protobuf"],
               check=True)

print("✅  All packages installed – restart the runtime now, then run Cell 2 onwards.")

✅  All packages installed – restart the runtime now, then run Cell 2 onwards.


In [ ]:
# Mount Google Drive & login to HuggingFace
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import login
HF_TOKEN = "hf_********************"   
login(HF_TOKEN)
print("✅  Drive mounted & HF login complete.")

Mounted at /content/drive
✅  Drive mounted & HF login complete.


In [ ]:
# Imports & reproducibility seeds

import os, json, random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅  Imports done.  GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


✅  Imports done.  GPU: NVIDIA A100-SXM4-80GB


In [ ]:
# Project paths

BASE_MODEL_ID  = "microsoft/Phi-4-mini-instruct"
HF_DATASET_ID  = "darklord1611/legal_citations"

PROJECT_DIR    = "/content/drive/MyDrive/phi4_thesis_HuggingFace"
SPLITS_DIR     = os.path.join(PROJECT_DIR, "splits")
QLORA_DIR      = os.path.join(PROJECT_DIR, "qlora_model")
METRICS_DIR    = os.path.join(PROJECT_DIR, "metrics")
PREDICTIONS_DIR = os.path.join(PROJECT_DIR, "predictions")

for folder in [PROJECT_DIR, SPLITS_DIR, QLORA_DIR, METRICS_DIR, PREDICTIONS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✅  Directories ready under:", PROJECT_DIR)

✅  Directories ready under: /content/drive/MyDrive/phi4_thesis_HuggingFace


In [ ]:

# Load dataset from HuggingFace & inspect
print(f"Loading dataset: {HF_DATASET_ID} …")
raw = load_dataset(HF_DATASET_ID)

print("\nAvailable splits :", list(raw.keys()))
print("Train rows       :", len(raw["train"]))
print("Test  rows       :", len(raw["test"]))
print("Columns          :", raw["train"].column_names)
print("\nSample row:")
print(raw["train"][0])


Loading dataset: darklord1611/legal_citations …


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/58.7M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21237 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3748 [00:00<?, ? examples/s]


Available splits : ['train', 'test']
Train rows       : 21237
Test  rows       : 3748
Columns          : ['Unnamed: 0', 'case_id', 'case_outcome', 'case_title', 'case_text']

Sample row:
{'Unnamed: 0': 21382, 'case_id': 'Case21569', 'case_outcome': 'distinguished', 'case_title': 'Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17', 'case_text': "The second issue to which I referred, can best be considered by reference to Housing Guarantee Fund Ltd v Yusef [1991] 2 VR 17 (' HGFL v Yusef ')."}


In [ ]:
# Convert to DataFrames & normalise column names

train_df_raw = raw["train"].to_pandas()
test_df_raw  = raw["test"].to_pandas()

# The HF dataset uses these column names (verified from card):
TEXT_COL  = "case_text"
LABEL_COL = "case_outcome"

# Defensive rename if the dataset uses slightly different names
for df in [train_df_raw, test_df_raw]:
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Keep only what we need
for df in [train_df_raw, test_df_raw]:
    for col in [TEXT_COL, LABEL_COL]:
        assert col in df.columns, \
            f"Column '{col}' not found. Available: {df.columns.tolist()}"

train_df_raw = train_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
test_df_raw  = test_df_raw[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

# Normalise text & labels
for df in [train_df_raw, test_df_raw]:
    df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

print("Raw train shape:", train_df_raw.shape)
print("Raw test  shape:", test_df_raw.shape)
print("\nLabel distribution (train):")
print(train_df_raw[LABEL_COL].value_counts().to_string())

Raw train shape: (21083, 2)
Raw test  shape: (3726, 2)

Label distribution (train):
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84


In [ ]:
#   Class-imbalance correction
#   Rules (applied to TRAIN only, then val/test are filtered to match):
#     • Drop any label whose total count in train < MIN_LABEL_COUNT
#     • Cap any label whose count exceeds MAX_LABEL_COUNT  (random undersample)

MIN_LABEL_COUNT = 100    # remove rare classes
MAX_LABEL_COUNT = 5000   # cap dominant classes

label_counts = train_df_raw[LABEL_COL].value_counts()
print("\n── Before balancing (train label counts) ──")
print(label_counts.to_string())

# 1) Remove labels below threshold
valid_labels = label_counts[label_counts >= MIN_LABEL_COUNT].index.tolist()
print(f"\nLabels kept  (≥{MIN_LABEL_COUNT} rows) : {valid_labels}")
removed = label_counts[label_counts < MIN_LABEL_COUNT].index.tolist()
print(f"Labels dropped (<{MIN_LABEL_COUNT} rows): {removed}")

train_df_filtered = train_df_raw[train_df_raw[LABEL_COL].isin(valid_labels)].copy()

# 2) Cap oversized labels
def cap_label_group(group, max_count, seed):
    if len(group) > max_count:
        return group.sample(n=max_count, random_state=seed)
    return group

train_df_balanced = (
    train_df_filtered
    .groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))
    .reset_index(drop=True)
)

print("\n After balancing (train label counts) ")
print(train_df_balanced[LABEL_COL].value_counts().to_string())
print("\nBalanced train shape:", train_df_balanced.shape)

# Keep LABELS list (sorted for reproducibility)
LABELS = sorted(train_df_balanced[LABEL_COL].unique().tolist())
print("\nFinal label set:", LABELS)


── Before balancing (train label counts) ──
case_outcome
cited            10296
referred to       3727
applied           2051
followed          1923
considered        1449
discussed          860
distinguished      505
approved            97
related             91
affirmed            84

Labels kept  (≥100 rows) : ['cited', 'referred to', 'applied', 'followed', 'considered', 'discussed', 'distinguished']
Labels dropped (<100 rows): ['approved', 'related', 'affirmed']

── After balancing (train label counts) ──
case_outcome
cited            5000
referred to      3727
applied          2051
followed         1923
considered       1449
discussed         860
distinguished     505

Balanced train shape: (15515, 2)

Final label set: ['applied', 'cited', 'considered', 'discussed', 'distinguished', 'followed', 'referred to']


/tmp/ipykernel_568/2244489308.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: cap_label_group(g, MAX_LABEL_COUNT, SEED))


In [ ]:
# Split HF test split → validation + test (50 / 50, stratified)
# ===========================================================================
# Only keep rows whose label is in the balanced training set
test_df_filtered = test_df_raw[test_df_raw[LABEL_COL].isin(LABELS)].reset_index(drop=True)

label_counts_test = test_df_filtered[LABEL_COL].value_counts()
# Stratify only when all classes have ≥ 2 samples in the pool
can_stratify = label_counts_test.min() >= 2

val_df, test_df = train_test_split(
    test_df_filtered,
    test_size=0.50,
    random_state=SEED,
    stratify=test_df_filtered[LABEL_COL] if can_stratify else None,
)

val_df  = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape      :", train_df_balanced.shape)
print("Validation shape :", val_df.shape)
print("Test shape       :", test_df.shape)
print("\nValidation label distribution:")
print(val_df[LABEL_COL].value_counts().to_string())
print("\nTest label distribution:")
print(test_df[LABEL_COL].value_counts().to_string())

# Save splits for later evaluation
train_df_balanced.to_csv(os.path.join(SPLITS_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(SPLITS_DIR, "val.csv"),   index=False)
test_df.to_csv(os.path.join(SPLITS_DIR, "test.csv"),  index=False)
# Also save the label list so evaluation script can reuse it
with open(os.path.join(SPLITS_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

print("\n✅  Splits saved to:", SPLITS_DIR)

Train shape      : (15515, 2)
Validation shape : (1836, 2)
Test shape       : (1836, 2)

Validation label distribution:
case_outcome
cited            907
referred to      318
applied          193
followed         165
considered       125
discussed         79
distinguished     49

Test label distribution:
case_outcome
cited            907
referred to      318
applied          194
followed         164
considered       125
discussed         79
distinguished     49

✅  Splits saved to: /content/drive/MyDrive/phi4_thesis_HuggingFace/splits


In [ ]:
# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=True,
)

tokenizer.model_max_length = 2048
# Phi-4-mini-instruct does not have a dedicated pad token;
# using eos_token avoids shape mismatches during batch padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"   # right-pad during training

print("Vocab size :", tokenizer.vocab_size)
print("Pad token  :", repr(tokenizer.pad_token))
print("EOS token  :", repr(tokenizer.eos_token))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Vocab size : 200019
Pad token  : '<|endoftext|>'
EOS token  : '<|endoftext|>'


In [ ]:
# Build prompt / chat-template formatting function

SYSTEM_MSG = (
    "You are a legal assistant specialising in case-outcome classification. "
    "Analyse the provided case text and return exactly one label from the "
    "allowed list — nothing else."
)

LABEL_STR = ", ".join(LABELS)   # e.g. "affirmed, applied, cited, ..."

def format_example(example):
    """
    Wraps a single row into Phi-4-mini-instruct chat format.
    The assistant turn contains ONLY the normalised label (ground truth).
    """
    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {
            "role": "user",
            "content": (
                f"Case Text:\n{example[TEXT_COL]}\n\n"
                f"Allowed labels: {LABEL_STR}.\n"
                "What is the outcome of this case? Return only the label."
            ),
        },
        {
            "role": "assistant",
            "content": str(example[LABEL_COL]).strip().lower(),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

In [ ]:
# Create HuggingFace Dataset objects & apply template

raw_ds = DatasetDict({
    "train":      Dataset.from_pandas(train_df_balanced),
    "validation": Dataset.from_pandas(val_df),
})

processed_train = raw_ds["train"].map(
    format_example,
    remove_columns=raw_ds["train"].column_names,
    desc="Formatting train",
)
processed_val = raw_ds["validation"].map(
    format_example,
    remove_columns=raw_ds["validation"].column_names,
    desc="Formatting validation",
)

print("Processed train size :", len(processed_train))
print("Processed val size   :", len(processed_val))
print("\n── Sample formatted text (first 800 chars) ──")
print(processed_train[0]["text"][:800])


Formatting train:   0%|          | 0/15515 [00:00<?, ? examples/s]

Formatting validation:   0%|          | 0/1836 [00:00<?, ? examples/s]

Processed train size : 15515
Processed val size   : 1836

── Sample formatted text (first 800 chars) ──
<|system|>You are a legal assistant specialising in case-outcome classification. Analyse the provided case text and return exactly one label from the allowed list — nothing else.<|end|><|user|>Case Text:
It is sufficient to note that intermediate courts have repeatedly been willing to hold that such a duty exists. However, as noted earlier I was referred to only one decision where the existence of such a duty (express or implied) was held to have been breached: Pacific Brands Sport &amp; Leisure Pty Ltd v Underworks Pty Ltd [2005] FCA 288 at [66] , a decision which Gyles J subsequently described as ' adventurous ': Council of the City of Sydney v Goldspar Australia Pty Ltd (2006) 230 ALR 437 at [166]. Two things should be noted about the decision at first instance. First, the breach of suc


In [ ]:
# 4-bit quantisation config (QLoRA)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # Normal-Float 4 (QLoRA paper default)
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,      # saves ~0.4 bits/param extra
)

print("✅  BitsAndBytes config ready.")

✅  BitsAndBytes config ready.


In [ ]:
# Load model in 4-bit & prepare for k-bit training

print(f"Loading {BASE_MODEL_ID} in 4-bit …  (may take 1-2 min)")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,              # must be False with gradient checkpointing
    attn_implementation="eager",  # safest for Phi-4-mini on Colab
)

model = prepare_model_for_kbit_training(model)

print("✅  Model loaded.  Trainable params (before LoRA):",
      sum(p.numel() for p in model.parameters() if p.requires_grad))


Loading microsoft/Phi-4-mini-instruct in 4-bit …  (may take 1-2 min)


config.json: 0.00B [00:00, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✅  Model loaded.  Trainable params (before LoRA): 0


In [ ]:
#   LoRA / PEFT config
#
#   r=32, alpha=64  → higher rank gives the adapter more capacity to learn
#   classification nuances across 8-10 legal outcome classes.
#   target_modules="all-linear" covers q, k, v, o, ffn projections in Phi-4.

peft_config = LoraConfig(
    r=32,                   # adapter rank  (was 16; 32 gives better capacity)
    lora_alpha=64,          # scaling factor = alpha/r = 2  (standard ratio)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",   # adapts every linear layer
)

print("✅  LoRA config ready.  r =", peft_config.r, "| alpha =", peft_config.lora_alpha)

✅  LoRA config ready.  r = 32 | alpha = 64


In [ ]:
#   SFT training config
#
#   EPOCH CHOICE — why 1 epoch is the right call on Colab A100:
#   • ~15 k–20 k balanced training samples → each of 7 labels is seen
#     2 000–5 000 times per epoch.  That is ample signal for a model that
#     is already instruction-tuned (Phi-4-mini-instruct).
#   • Estimated runtime on A100: ~3–4 h for 1 epoch (legal texts are long).
#     2 epochs ≈ 6–8 h, risking Colab disconnection and Drive quota issues.
#   • QLoRA adapters converge faster than full fine-tuning; overfitting is a
#     larger risk than underfitting here.
#   • TIP: if val-loss is still dropping sharply at the end of epoch 1, you
#     can resume from the last checkpoint and run a second epoch; the
#     save_total_limit=2 keeps the best checkpoint safe on Drive.
#
#   KEY FIXES vs. previous version:
#     • max_steps=-1  →  train for the full num_train_epochs (no step cap)
#     • num_train_epochs=1  →  1 full pass; sufficient + Colab-safe
#     • learning_rate=2e-4  →  QLoRA paper sweet-spot; cosine decay
#     • warmup_ratio=0.05   →  ~5% of steps for learning-rate warm-up
#     • optim="paged_adamw_8bit"  →  memory-efficient QLoRA optimiser
#     • load_best_model_at_end=True  →  saves best checkpoint automatically
# ===========================================================================

# Compute a sensible eval/save cadence dynamically so it works regardless
# of the exact post-balancing dataset size (avoids hard-coding 100/500).
# Target: evaluate ~5 times per epoch; save at each eval point.
# We estimate steps conservatively; the actual value is printed below.
_ESTIMATED_TRAIN_ROWS   = len(train_df_balanced)
_PER_DEVICE_BATCH       = 2
_GRAD_ACCUM             = 4
_EFFECTIVE_BATCH        = _PER_DEVICE_BATCH * _GRAD_ACCUM           # = 8
_STEPS_PER_EPOCH_EST    = max(1, _ESTIMATED_TRAIN_ROWS // _PER_DEVICE_BATCH)
_EVAL_EVERY_N_STEPS     = 300
_SAVE_EVERY_N_STEPS     = _EVAL_EVERY_N_STEPS                       # save == eval

print(f"Estimated steps/epoch : {_STEPS_PER_EPOCH_EST}")
print(f"Eval / save every     : {_EVAL_EVERY_N_STEPS} steps")

sft_config = SFTConfig(
    #  Output 
    output_dir=QLORA_DIR,

    #  Precision 
    bf16=True,

    #  Epochs & steps 
    num_train_epochs=1,   # 1 full epoch — safe on Colab A100 (~3–4 h)
    max_steps=-1,         # -1 = honour num_train_epochs (no step cap)

    #  Batch size & gradient accumulation 
    per_device_train_batch_size=_PER_DEVICE_BATCH,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=_GRAD_ACCUM,   # effective batch = 8

    #  Learning rate schedule 
    learning_rate=2e-4,         # QLoRA paper recommendation
    lr_scheduler_type="cosine", # smooth decay; best for single-epoch runs
    warmup_ratio=0.05,          # 5% of steps for warm-up

    #  Gradient checkpointing (saves ~30% VRAM, small speed trade-off) 
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    #  Optimiser (paged = offloads Adam states; essential for 4-bit QLoRA) 
    optim="paged_adamw_8bit",

    #  Logging 
    logging_strategy="steps",
    logging_steps=20,           # print loss every 20 steps
    report_to="none",

    #  Evaluation 
    do_eval=True,
    eval_strategy="steps",
    eval_steps=_EVAL_EVERY_N_STEPS,

    #  Checkpointing 
    save_strategy="steps",
    save_steps=_SAVE_EVERY_N_STEPS,
    save_total_limit=2,         # keep the 2 best checkpoints on Drive

    #  Dataset 
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,              # keep off — packing mixes samples across labels
    remove_unused_columns=True,

    #  Best-model restoration 
    seed=SEED,
    load_best_model_at_end=True,   # restores lowest val-loss checkpoint before save
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print("\n✅  SFTConfig ready.")
print(f"   Epochs     : {sft_config.num_train_epochs}  (set max_steps=-1 to honour this)")
print(f"   max_steps  : {sft_config.max_steps}")
print(f"   LR         : {sft_config.learning_rate}")
print(f"   Eff. batch : {_PER_DEVICE_BATCH} × {_GRAD_ACCUM} accum = {_EFFECTIVE_BATCH}")

Estimated steps/epoch : 7757
Eval / save every     : 300 steps

✅  SFTConfig ready.
   Epochs     : 1  (set max_steps=-1 to honour this)
   max_steps  : -1
   LR         : 0.0002
   Eff. batch : 2 × 4 accum = 8


In [ ]:
# Build SFTTrainer & start training

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    peft_config=peft_config,
    train_dataset=processed_train,
    eval_dataset=processed_val,
    processing_class=tokenizer,   # replaces deprecated 'tokenizer=' in trl≥0.12
)

print(f"Trainable parameters after LoRA: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n🚀  Starting training …")
train_result = trainer.train()
train_metrics = train_result.metrics
print("\n── Training metrics ──")
print(json.dumps(train_metrics, indent=2))

Converting train dataset to ChatML:   0%|          | 0/15515 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15515 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/1836 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1836 [00:00<?, ? examples/s]

Trainable parameters after LoRA: 46,137,344

🚀  Starting training …


Step,Training Loss,Validation Loss
300,1.845000,1.806594
600,1.748200,1.728904
900,1.672400,1.662167
1200,1.644400,1.614149
1500,1.602500,1.580316
1800,1.551700,1.567023



── Training metrics ──
{
  "train_runtime": 6897.7825,
  "train_samples_per_second": 2.249,
  "train_steps_per_second": 0.281,
  "total_flos": 2.378923649630331e+17,
  "train_loss": 1.7183211564156984
}


In [ ]:
# Save model, tokenizer & metrics

# save_model() saves the LoRA adapter weights (not the full 14B base)
trainer.save_model(QLORA_DIR)
tokenizer.save_pretrained(QLORA_DIR)

# Save label list alongside the adapter so the evaluation script is self-contained
with open(os.path.join(QLORA_DIR, "labels.json"), "w") as f:
    json.dump(LABELS, f, indent=2)

# Persist training metrics
with open(os.path.join(METRICS_DIR, "train_metrics.json"), "w") as f:
    json.dump(train_metrics, f, indent=2)

print("✅  Model adapter saved to:", QLORA_DIR)
print("   Files in QLORA_DIR:", os.listdir(QLORA_DIR))

✅  Model adapter saved to: /content/drive/MyDrive/phi4_thesis_HuggingFace/qlora_model
   Files in QLORA_DIR: ['checkpoint-1800', 'checkpoint-1939', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'training_args.bin', 'labels.json']


In [ ]:
# Final validation-loss evaluation & save
# ===========================================================================
eval_metrics = trainer.evaluate()
eval_metrics["eval_samples"] = len(processed_val)

print("\n── Validation metrics ──")
print(json.dumps(eval_metrics, indent=2))

with open(os.path.join(METRICS_DIR, "eval_loss_metrics.json"), "w") as f:
    json.dump(eval_metrics, f, indent=2)

print("\n✅  Fine-tuning complete.")
print("    QLoRA adapter :", QLORA_DIR)
print("    Splits        :", SPLITS_DIR)
print("    Metrics       :", METRICS_DIR)
print("\nNext step: run the evaluation / inference script on the held-out test set.")



── Validation metrics ──
{
  "eval_loss": 1.5670230388641357,
  "eval_runtime": 183.4492,
  "eval_samples_per_second": 10.008,
  "eval_steps_per_second": 5.004,
  "eval_samples": 1836
}

✅  Fine-tuning complete.
    QLoRA adapter : /content/drive/MyDrive/phi4_thesis_HuggingFace/qlora_model
    Splits        : /content/drive/MyDrive/phi4_thesis_HuggingFace/splits
    Metrics       : /content/drive/MyDrive/phi4_thesis_HuggingFace/metrics

Next step: run the evaluation / inference script on the held-out test set.


In [ ]:
from google.colab import runtime
runtime.unassign()